
# PageIndex Agentic RAG — Colab Notebook

This notebook demonstrates how to connect **PageIndex** to an agent and use its document tools for:

- Document browsing
- Document search
- Page search
- Document structure inspection
- Page content retrieval
- Page image / zoom workflows
- MCP-based integration
- Page-level citations
- A hybrid architecture with Milvus/Weaviate

> **Goal:** build a practical PageIndex-powered document agent that keeps its own LLM and uses PageIndex as a document intelligence/tool layer.



## Architecture

```text
                         User Question
                               |
                               v
                         AI Agent / LLM
                               |
                    +----------+----------+
                    |                     |
                    v                     v
               PageIndex MCP         Vector RAG
                    |                Milvus/Weaviate
                    v                     |
             Document Intelligence        |
                    |                     |
          +---------+---------+           |
          |         |         |           |
        Search    Read      Zoom          |
          |         |         |           |
          +---------+---------+-----------+
                    |
                    v
              Evidence Fusion
                    |
                    v
             Final Answer
                    |
                    v
          Page / Region Citation
```


In [ ]:

# Install PageIndex
!pip -q install pageindex


In [ ]:
import os
from google.colab import userdata

try:
    os.environ["PAGEINDEX_API_KEY"] = userdata.get("PAGEINDEX_API_KEY")
except Exception as e:
    print("Could not retrieve PAGEINDEX_API_KEY from secrets:", e)

print("PAGEINDEX_API_KEY configured:", bool(os.getenv("PAGEINDEX_API_KEY")))

PAGEINDEX_API_KEY configured: True



## 1. Create a PageIndex client

When PageIndex is used as a tool provider for an existing agent, the agent can keep its own model.

The client does not need to specify `chat` in this mode.


In [ ]:

from pageindex import PageIndexClient

client = PageIndexClient(
    index="cloud",
)

print("PageIndex client created.")


PageIndex client created.



## 2. Upload a PDF

Upload a PDF to Colab and send it to PageIndex.

The exact upload method can vary with the installed PageIndex SDK version, so inspect the available client methods first.


In [ ]:

# Inspect available client methods
[m for m in dir(client) if not m.startswith("_")][:100]


['BASE_URL',
 'agent_instructions',
 'agent_tools',
 'anthropic_runner_config',
 'api_key',
 'as_anthropic_tools',
 'as_claude_mcp',
 'as_openai_tools',
 'build_outline_index',
 'chat',
 'chat_backend',
 'chat_completions',
 'chat_model',
 'citation_prompt',
 'claude_agent_config',
 'collection_context',
 'create_collection',
 'create_folder',
 'create_metadata_schema',
 'delete_collection',
 'delete_document',
 'delete_metadata_schema',
 'document_context',
 'folder_context',
 'get_block',
 'get_citations',
 'get_collection',
 'get_document',
 'get_document_id',
 'get_document_image',
 'get_document_path',
 'get_document_structure',
 'get_folder_id',
 'get_folder_path',
 'get_metadata_schema',
 'get_ocr',
 'get_page_content',
 'get_page_image',
 'get_retrieval',
 'get_tree',
 'instructions',
 'is_retrieval_ready',
 'list_collections',
 'list_documents',
 'list_folders',
 'openai_agent_config',
 'page_image',
 'resolve_citations',
 'retrieve_model',
 'submit_document',
 'submit_query',

In [ ]:

from google.colab import files

uploaded = files.upload()
pdf_name = next(iter(uploaded))

print("Uploaded:", pdf_name)


Saving bryce-canyon-map.pdf to bryce-canyon-map.pdf
Uploaded: bryce-canyon-map.pdf



## 3. Find the document management methods

Use the SDK's introspection to identify the upload method exposed by the installed version.


In [ ]:

# Look for likely document/upload methods
[m for m in dir(client) if any(x in m.lower() for x in ["upload", "document", "file"])]


['_upload',
 'delete_document',
 'document_context',
 'get_document',
 'get_document_id',
 'get_document_image',
 'get_document_path',
 'get_document_structure',
 'list_documents',
 'submit_document',
 'update_document',
 'wait_document']


## 4. Agent context

Once a document has been uploaded and its `doc_id` is known, create focused context for the agent.

Example:

```python
context = client.document_context(doc_id)

messages = [
    {"role": "user", "content": context},
    {"role": "user", "content": "What are the auditor's main concerns?"}
]
```

The context guides the agent but does not necessarily prevent it from using other PageIndex documents.


In [ ]:
# Submit the uploaded PDF to PageIndex to obtain a valid document ID
print(f"Uploading {pdf_name} to PageIndex...")
submit_result = client.submit_document(pdf_name)

# Extract the correct document ID from 'doc_id', 'id', or 'document_id'
doc_id = submit_result.get("doc_id") or submit_result.get("id") or submit_result.get("document_id")
print(f"Document submitted successfully. ID: {doc_id}")

# Ensure the document is ready for context retrieval
print("Waiting for document processing...")
client.wait_document(doc_id)

# Retrieve document context
context = client.document_context(doc_id)

messages = [
    {"role": "user", "content": context},
    {"role": "user", "content": "What are the key findings in this document?"}
]

print("Context successfully retrieved! Preview:")
print(messages[0]["content"][:2000])

Uploading bryce-canyon-map.pdf to PageIndex...
Document submitted successfully. ID: pi-cmwxsvklb00360cpdmbsiil9u
Waiting for document processing...
Context successfully retrieved! Preview:
The user has specified document: bryce-canyon-map.pdf
Document metadata: {"id": "pi-cmwxsvklb00360cpdmbsiil9u", "name": "bryce-canyon-map.pdf", "description": null, "status": "completed", "indexStatus": "completed", "createdAt": "2026-11-26T17:02:10.972000", "pageNum": 1, "folderId": null, "collectionId": null, "metadata": {}, "path": "bryce-canyon-map.pdf"}
Use this document's name to retrieve its content with get_document_structure() and get_page_content().



## 5. Give PageIndex tools to an OpenAI Agents SDK agent

This is the key agentic pattern:

**Your agent owns the model and reasoning loop. PageIndex supplies document tools.**


In [ ]:

# Install the OpenAI Agents SDK if needed
!pip -q install openai-agents


In [ ]:

from agents import Agent, Runner

instructions = client.agent_instructions()

agent = Agent(
    name="PageIndex Document Agent",
    instructions=instructions,
    tools=client.as_openai_tools(),
    model="gpt-6.2-sol",
)

print("Agent configured.")


Agent configured.


In [ ]:
import os
from google.colab import userdata

# Ensure both API keys are configured in the environment
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception as e:
    print("Could not retrieve OPENAI_API_KEY from secrets:", e)

try:
    os.environ["PAGEINDEX_API_KEY"] = userdata.get("PAGEINDEX_API_KEY")
except Exception as e:
    print("Could not retrieve PAGEINDEX_API_KEY from secrets:", e)

# Re-instantiate the agent with a standard, accessible model like 'gpt-4o'
from agents import Agent, Runner

agent = Agent(
    name="PageIndex Document Agent",
    instructions=client.agent_instructions(),
    tools=client.as_openai_tools(),
    model="gpt-4o",
)

# Run the agent asynchronously
result = await Runner.run(
    agent,
    messages
)

print(result.final_output)

The document, "bryce-canyon-map.pdf," is a map of Bryce Canyon National Park. Key features include:

- **Trails and Campgrounds:** The map outlines various trails, campgrounds, and key geographical features within Bryce Canyon.
- **Notable Points:** Includes locations such as Fairyland Point, Sunrise Point, Sunset Point, and Inspiration Point.
- **Facilities:** Symbols for restrooms, shuttle bus stops, picnic areas, and ranger stations are included.
- **Natural Features:** Highlights natural landmarks like Natural Bridge, Agua Canyon, and Rainbow Point.
- **Distances:** Provides distance markers and elevation information in various parts of the park.

To explore any specific area or feature in detail, I can zoom in on particular sections of the map. Let me know if you'd like more information on any part!



## 6. Enable page + region citations

PageIndex can instruct the agent to cite claims with the source document, page and bounding box.

```python
instructions = (
    client.agent_instructions()
    + "\n\n"
    + client.citation_prompt("cite_with_boxes")
)
```

A citation can look like:

```text
Revenue increased during the reporting period.
<cite doc="report.pdf" page="12" box="0.08,0.31,0.92,0.44"/>
```


In [ ]:
instructions = (
    client.agent_instructions()
    + "\n\n"
    + client.citation_prompt("cite_with_boxes")
)

agent = Agent(
    name="Cited PageIndex Agent",
    instructions=instructions,
    tools=client.as_openai_tools(),
    model="gpt-4o",
)

# Run the agent asynchronously to bypass the active event loop error
result = await Runner.run(
    agent,
    messages
)

print(result.final_output)

The document "bryce-canyon-map.pdf" contains a map of Bryce Canyon National Park. Key features include:

- Overlook and trail locations.
- Distance indicators.
- Restrooms, amphitheater, ranger stations, campgrounds, and shuttle bus stops.
- Notable points like Fairyland Point, Inspiration Point, and Bryce Point.
- Trails such as Fairyland Loop and Under-the-Rim Trail.
- Major roads and connecting trails within the park.
- Various canyon points, including Paria and Agua Canyon.
- Key geographical markers, such as elevation indicators and surrounding forests.

The map provides a detailed layout of the park, highlighting amenities and paths for visitors.


In [ ]:

# Resolve PageIndex citations into structured citation metadata
resolved = client.resolve_citations(result.final_output)

resolved


{'answer': 'The document "bryce-canyon-map.pdf" contains a map of Bryce Canyon National Park. Key features include:\n\n- Overlook and trail locations.\n- Distance indicators.\n- Restrooms, amphitheater, ranger stations, campgrounds, and shuttle bus stops.\n- Notable points like Fairyland Point, Inspiration Point, and Bryce Point.\n- Trails such as Fairyland Loop and Under-the-Rim Trail.\n- Major roads and connecting trails within the park.\n- Various canyon points, including Paria and Agua Canyon.\n- Key geographical markers, such as elevation indicators and surrounding forests.\n\nThe map provides a detailed layout of the park, highlighting amenities and paths for visitors.',
 'citations': []}


## 7. Inspect cited page regions

For citations containing a bounding box, PageIndex can return the cited page image or a cropped region.

```python
url = client.get_page_image(doc_id, page)
crop = client.page_image(doc_id, page, box=box)
```

This is useful for audit, compliance and human verification.



## 8. Direct PageIndex chat

PageIndex can also own the agent loop if you want a quick standalone document Q&A test:

```python
answer = client.chat(
    "What are the key findings in this document?",
    doc_id=doc_id,
    citations="cite_with_boxes",
)

print(answer)
```

This is useful for benchmarking PageIndex against your own agent implementation.


In [ ]:

# Optional direct PageIndex test
# answer = client.chat(
#     "What are the key findings in this document?",
#     doc_id=doc_id,
#     citations="cite_with_boxes",
# )
# print(answer)



## 9. MCP architecture

PageIndex can also be connected as an MCP server.

Endpoint:

```text
https://api.pageindex.ai/mcp
```

Read-only mode:

```text
https://api.pageindex.ai/mcp?tools=read
```

Conceptually:

```text
Agent Runtime
     |
     | MCP
     v
PageIndex MCP Server
     |
     +-- browse documents
     +-- search documents
     +-- inspect structure
     +-- search pages
     +-- read pages
     +-- view pages
     +-- zoom regions
```

For production, store the API key in a secret manager and use folder scope to enforce tenant boundaries.


### Try Document Q&A

Run the cell below, type in your question, and get answers directly from your document.

In [ ]:
question = "Which page contains the revenue table?" # @param {type:"string"}

# Configure user prompt with document context
qa_messages = [
    {"role": "user", "content": context},
    {"role": "user", "content": question}
]

# Run the agent to query the document
response = await Runner.run(
    agent,
    qa_messages
)

print("=== Question ===")
print(question)
print("\n=== Answer ===")
print(response.final_output)

=== Question ===
Which page contains the revenue table?

=== Answer ===
The document "bryce-canyon-map.pdf" does not contain a revenue table. It seems to primarily consist of a map of Bryce Canyon National Park.



## 10. Hybrid PageIndex + Vector RAG

For an enterprise system, combine PageIndex with your existing vector infrastructure.

```text
                    Question
                       |
                 Agent Router
                       |
          +------------+------------+
          |                         |
          v                         v
     Vector Search             PageIndex
    Milvus/Weaviate          Document Agent
          |                         |
          +------------+------------+
                       |
                 Evidence Fusion
                       |
                    Reranker
                       |
                 Enterprise
                  LLM Gateway
                       |
                    Answer
                       |
              Page-level citation
```

Recommended routing:

| Question | Preferred path |
|---|---|
| Simple semantic lookup | Vector DB |
| Long complex PDF | PageIndex |
| Financial filing | PageIndex |
| Scanned document | PageIndex |
| Table-heavy document | PageIndex |
| Cross-document semantic search | Hybrid |
| Structured enterprise data | SQL/API |



## 11. Benchmark plan

Measure the same question set across:

1. Traditional vector RAG
2. PageIndex
3. PageIndex + vector hybrid

Track:

- Answer accuracy
- Citation accuracy
- Page retrieval accuracy
- Table retrieval accuracy
- Scan/diagram retrieval accuracy
- Latency
- Token usage
- Cost per question
- Human verification time

### Suggested benchmark questions

```text
Q1: What are the three major findings?

Q2: Which page contains the revenue table?

Q3: What changed compared with the previous year?

Q4: What risks did the auditor identify?

Q5: Find the exact figure supporting the revenue claim.

Q6: Which section contains the recommendation?

Q7: Extract the table into structured JSON.
```



# Final architecture for Bazai Agent Paradise

```text
                         BAZAI AGENT PARADISE
                                  |
                           Agent Runtime
                                  |
                         MCP Tool Gateway
                                  |
        +-------------------------+-------------------------+
        |                         |                         |
        v                         v                         v
     PageIndex                 Milvus                  Enterprise APIs
   Document Agent           Vector Search              SQL / CRM / Git
        |                         |
        +------------+------------+
                     |
               Evidence Fusion
                     |
                LLM Gateway
                     |
          OpenAI / Anthropic / Gemini
                     |
                     v
              Agent Response
                     |
            +--------+--------+
            |                 |
         Answer          Evidence/Citation
```

**Core design principle:**

> PageIndex should be treated as a specialized document-intelligence tool inside the agent runtime, not as a replacement for your entire enterprise RAG platform.
